# 🛡️ Comprehensive Credit Card Fraud Detection Analysis
### Course: CS506 - Big Data Analysis & Machine Learning
### Project: Enterprise-Grade Fraud Shield System
---

## 1. Introduction & Vision
In this project, we address the critical problem of financial fraud. Using a dataset of over 284,000 transactions, our goal is to build a robust model that can identify suspicious patterns with high precision and recall. 

### Objectives:
- Exploratory Data Analysis (EDA) to understand fraud patterns.
- Addressing class imbalance (SMOTE).
- Model Training & Real-time Inference simulation.
- Integration with AI for security insights.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
import joblib

import warnings
warnings.filter_material = 'ignore'

plt.style.use('fivethirtyeight')
print("✅ Libraries Loaded Successfully!")

## 2. Data Loading & Initial Inspection
We use the Credit Card Fraud dataset from Kaggle, which contains transactions made by credit cards in September 2013 by European cardholders.

In [ ]:
df = pd.read_csv('D:/creditcard.csv')
print(f"Dataset Dimension: {df.shape[0]} rows, {df.shape[1]} columns")
df.info()

## 3. Exploratory Data Analysis (EDA)
Visualizing the data distribution is crucial for Big Data projects.

In [ ]:
# Visualizing the Class Imbalance
plt.figure(figsize=(10, 6))
sns.countplot(x='Class', data=df, palette='viridis')
plt.title('Transaction Distribution (0: Normal, 1: Fraud)')
plt.yscale('log')
plt.show()

In [ ]:
# Distribution of Amount and Time
fig, ax = plt.subplots(1, 2, figsize=(18,4))

amount_val = df['Amount'].values
time_val = df['Time'].values

sns.distplot(amount_val, ax=ax[0], color='r')
ax[0].set_title('Distribution of Transaction Amount', fontsize=14)
ax[0].set_xlim([min(amount_val), max(amount_val)])

sns.distplot(time_val, ax=ax[1], color='b')
ax[1].set_title('Distribution of Transaction Time', fontsize=14)
ax[1].set_xlim([min(time_val), max(time_val)])

plt.show()

## 4. Preprocessing & Feature Engineering
Scaling numerical features like 'Time' and 'Amount' and handling the target variable.

In [ ]:
scaler = StandardScaler()
df[['scaled_amount', 'scaled_time']] = scaler.fit_transform(df[['Amount', 'Time']])
df = df.drop(['Time', 'Amount'], axis=1)

X = df.drop('Class', axis=1)
y = df['Class']

## 5. Handling Imbalance (SMOTE)
Since fraud cases are very low (0.17%), we use Synthetic Minority Over-sampling Technique (SMOTE) to balance the classes.

In [ ]:
sm = SMOTE(random_state=42)
X_res, y_res = sm.fit_resample(X, y)
print(f"After SMOTE, Class 0 count: {sum(y_res==0)}")
print(f"After SMOTE, Class 1 count: {sum(y_res==1)}")

## 6. Training & Evaluation
Using Random Forest for high-accuracy classification.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_res, y_res, test_size=0.2, random_state=42)
rf = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

## 7. Feature Importance Visualization
Understanding which features contribute most to fraud detection.

In [ ]:
importances = rf.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(12, 6))
plt.title("Feature Importances")
plt.bar(range(X.shape[1]), importances[indices], align="center")
plt.xticks(range(X.shape[1]), X.columns[indices], rotation=90)
plt.show()

## 8. Exporting for Production
Saving the trained model and scaler to joblib files.

In [ ]:
joblib.dump(rf, 'fraud_model.joblib')
joblib.dump(scaler, 'scaler.joblib')
print("🚀 Model and Scaler Ready for Streamlit Deployment!")